In [7]:
# Sparse Optical Flow - Lucas-Kanade

import cv2
import numpy as np

cap = cv2.VideoCapture(0)
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, maxCorners=100, qualityLevel=.3, minDistance=7, blockSize=7)
mask = np.zeros_like(old_frame)
lk = dict(winSize = (15,15), maxLevel = 2, criteria = (cv2.TermCriteria_EPS| cv2.TermCriteria_COUNT, 10, .03))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if p0 is None:
        p0 = cv2.goodFeaturesToTrack(gray, maxCorners=100, qualityLevel=.3, minDistance=7)
        mask[:]=0
    else:
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, gray, p0, None, **lk)
        if p1 is not None:
            good_new = p1[st==1]
            good_old = p0[st==1]
            for new, old in zip(good_new, good_old):
                x1, y1 = new.ravel().astype(int)
                x2, y2 = old.ravel().astype(int)
                cv2.line(mask, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.circle(frame, (x1, y1), 4, (0,0,255), -1)
            p0 = good_new.reshape(-1,1,2)
    output = cv2.add(frame, mask)
    cv2.imshow("Sparse Optical Flow", output)
    old_gray =gray
    if cv2.waitKey(1)&0xFF ==27:
        break
cap.release()
cv2.destroyAllWindows()

In [6]:
# Dense  = > farneback => visu => HSV

import cv2
import numpy as np

cap = cv2.VideoCapture(0)
ret, old = cap.read()
old_gray = cv2.cvtColor(old, cv2.COLOR_BGR2GRAY)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(old_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[...,0],flow[...,1])
    hsv = np.zeros_like(frame)
    # hue => direction of motion, saturation => intensity / purity of color, value => magnitude of motion
    hsv[...,1] = 255 # saturation
    hsv[...,0] = ang *180/np.pi/2 # hue => angle
    hsv[...,2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

    flow_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)    
    cv2.imshow("Dense Optical Flow", flow_bgr)
    old_gray =gray
    if cv2.waitKey(1)&0xFF ==27:
        break
cap.release()
cv2.destroyAllWindows()

In [9]:
# Estimate dominate motion direction  => dense optical flow

import cv2
import numpy as np

cap = cv2.VideoCapture(0)
ret, old = cap.read()
old_gray = cv2.cvtColor(old, cv2.COLOR_BGR2GRAY)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(old_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    fx, fy = flow[...,0], flow[...,1]
    mag, ang = cv2.cartToPolar(fx, fy)
    moving = mag >1.0
    if np.any(moving):
        dx = float(np.mean(fx[moving]))
        dy = float(np.mean(fy[moving]))
        angle = np.degrees(np.arctan2(dx, dy))
        text = f"Mean motion:{angle:.1f}deg"
    else:
        text = 'No significant motion'
    cv2.putText(frame, text,(20,40), cv2.FONT_HERSHEY_SIMPLEX, .8, (255,0,0),2)
    cv2.imshow("Motion Direction", frame)
    old_gray =gray
    if cv2.waitKey(1)&0xFF ==27:
        break
cap.release()
cv2.destroyAllWindows()

In [10]:
# Estimate dominate motion direction  => dense optical flow

import cv2
import numpy as np

cap = cv2.VideoCapture(0)

ret, old = cap.read()
if not ret:
    raise RuntimeError("Camera unavailable.")

old_gray = cv2.cvtColor(old, cv2.COLOR_BGR2GRAY)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow
    flow = cv2.calcOpticalFlowFarneback(
        old_gray, gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    fx = flow[..., 0]
    fy = flow[..., 1]

    # Draw optical-flow arrows
    step = 20

    for y in range(0, frame.shape[0], step):
        for x in range(0, frame.shape[1], step):

            dx = fx[y, x]
            dy = fy[y, x]

            # Draw only significant motion
            if np.sqrt(dx**2 + dy**2) > 1.0:

                cv2.arrowedLine(
                    frame,
                    (x, y),
                    (int(x + dx * 5), int(y + dy * 5)),
                    (0, 255, 0),
                    2,
                    tipLength=0.3
                )

    cv2.putText(
        frame,
        "Optical Flow",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.imshow("Optical Flow with Arrows", frame)

    old_gray = gray

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()